# Process Raw Data

In [1]:
import os
import json
from pathlib import Path
from pprint import pprint

from parser import *
from scaffolder import *


In [2]:
from pathlib import Path

folder = Path("input")
list_text = []
for file in sorted(folder.glob("*.txt")):
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()
        list_text.append(text)


In [3]:
# Step 1: Parser
for i in range(len(list_text)):
    list_text[i] = parser(list_text[i])

# Step 2: Build Tree
trees = []
for i, text in enumerate(list_text, start=1):
    trees.append(build_tree(text, doc_id=f"VB{i}"))

In [5]:
pprint(trees)

[Part(partID='VB1',
      text='1.  Tiền sử bệnh\n'
           'a. Thuốc trước khi nhập viện\n'
           'a1. metoprolol 25mg po bid\n'
           'a2. doxycycline cho viêm tuyến mồ hôi\n'
           'a3. atenolol (uống hôm nay)\n'
           'b. Các yếu tố nguy cơ liên quan\n'
           'b1. Theo lời bệnh nhân kể lại bệnh nhân có   Căng thẳng  nhiều '
           'trong công việc\n'
           'b2. Mất việc làm 8 ngày trước\n'
           'b3. Một ngày người bệnh có thể uống hàng chục tách cà phê có '
           'caffeine\n'
           'b4. và 1 tách cà phê không caffeine\n'
           '2.  Tiền sử bệnh hiện tại\n'
           'a. Lý do nhập viện:  Bệnh nhân vào viện vì xuất hiện triệu chứng '
           'đánh trống ngực\n'
           'b. Thời điểm khởi phát triệu chứng:\n'
           'b1. Cách 10 ngày trước khi vào việc  bệnh nhân xuất hiện cảm giác '
           'đánh trống ngực\n'
           'c. Các triệu chứng hiện tại\n'
           'c1. đánh trống ngực\n'
           'c2. Khó thở n

# Process in Neo4j Database

In [6]:
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = ""

driver = GraphDatabase.driver(URI,auth=(USERNAME, PASSWORD))

In [17]:
# Step 1: Create Entity
def create_part(tx, part: Part):
    if "_" not in part.partID:
        label = "Document"
    else:
        level = part.partID.count("_")
        if level == 1:
            label = "Chapter"
        elif level == 2:
            label = "Section"
        else:
            label = "Subsection"

    cypher = f"""
    MERGE (p:Part:{label} {{partID:$partID}})
    SET p.text=$text
    """
    tx.run(cypher, partID=part.partID, text=part.text,)

def create_relation(tx, parent_id, child_id):
    cypher = """
        MATCH (parent:Part {partID: $parent})
        MATCH (child:Part {partID: $child})
        MERGE (parent)-[:CHUA]->(child)
    """
    tx.run(cypher, parent=parent_id, child=child_id)

# Recursion node
def insert_part(tx, part: Part):
    create_part(tx, part)

    for child in part.child:
        insert_part(tx, child)
        create_relation(tx, part.partID, child.partID)

In [18]:
from tqdm import tqdm

with driver.session() as session:
    for tree in tqdm(trees, desc="Creating Neo4j trees", unit="doc"):
        session.execute_write(insert_part, tree)

print("Complete create all tree!")

Creating Neo4j trees:   2%|▏         | 2/100 [00:00<00:06, 15.60doc/s]

Document VB1
Chapter VB1_1
Section VB1_1_a
Subsection VB1_1_a_a1
Subsection VB1_1_a_a2
Subsection VB1_1_a_a3
Section VB1_1_b
Subsection VB1_1_b_b1
Subsection VB1_1_b_b2
Subsection VB1_1_b_b3
Subsection VB1_1_b_b4
Chapter VB1_2
Section VB1_2_a
Section VB1_2_b
Subsection VB1_2_b_b1
Section VB1_2_c
Subsection VB1_2_c_c1
Subsection VB1_2_c_c2
Subsection VB1_2_c_c3
Subsection VB1_2_c_c4
Subsection VB1_2_c_c5
Subsection VB1_2_c_c6
Subsection VB1_2_c_c7
Section VB1_2_d
Subsection VB1_2_d_d1
Subsection VB1_2_d_d2
Subsection VB1_2_d_d3
Subsection VB1_2_d_d4
Subsection VB1_2_d_d5
Subsection VB1_2_d_d6
Subsection VB1_2_d_d7
Section VB1_2_e
Subsection VB1_2_e_e1
Subsection VB1_2_e_e2
Subsection VB1_2_e_e3
Subsection VB1_2_e_e4
Subsection VB1_2_e_e5
Subsection VB1_2_e_e6
Subsection VB1_2_e_e7
Subsection VB1_2_e_e8
Subsection VB1_2_e_e9
Subsection VB1_2_e_e10
Subsection VB1_2_e_e11
Section VB1_2_f
Chapter VB1_3
Section VB1_3_a
Subsection VB1_3_a_a1
Subsection VB1_3_a_a2
Section VB1_3_b
Section VB1_3

Creating Neo4j trees:   9%|▉         | 9/100 [00:00<00:03, 23.18doc/s]

Document VB5
Chapter VB5_1
Section VB5_1_a
Chapter VB5_2
Section VB5_2_a
Section VB5_2_b
Subsection VB5_2_b_b1
Subsection VB5_2_b_b2
Section VB5_2_c
Subsection VB5_2_c_c1
Subsection VB5_2_c_c2
Subsection VB5_2_c_c3
Subsection VB5_2_c_c4
Section VB5_2_d
Subsection VB5_2_d_d1
Subsection VB5_2_d_d2
Subsection VB5_2_d_d3
Subsection VB5_2_d_d4
Subsection VB5_2_d_d5
Subsection VB5_2_d_d6
Subsection VB5_2_d_d7
Subsection VB5_2_d_d8
Subsection VB5_2_d_d9
Subsection VB5_2_d_d10
Chapter VB5_3
Section VB5_3_a
Section VB5_3_b
Section VB5_3_c
Document VB6
Chapter VB6_1
Section VB6_1_a
Subsection VB6_1_a_a1
Subsection VB6_1_a_a2
Subsection VB6_1_a_a3
Subsection VB6_1_a_a4
Subsection VB6_1_a_a5
Section VB6_1_b
Section VB6_1_c
Section VB6_1_d
Subsection VB6_1_d_d1
Subsection VB6_1_d_d2
Subsection VB6_1_d_d3
Subsection VB6_1_d_d4
Chapter VB6_2
Section VB6_2_a
Section VB6_2_b
Subsection VB6_2_b_b1
Subsection VB6_2_b_b2
Subsection VB6_2_b_b3
Subsection VB6_2_b_b4
Subsection VB6_2_b_b5
Section VB6_2_c
Sub

Creating Neo4j trees:  12%|█▏        | 12/100 [00:00<00:04, 21.43doc/s]

...

Subsection VB95_2_n_n2
Chapter VB95_3
Document VB96
Chapter VB96_1
Section VB96_1_a
Chapter VB96_2
Section VB96_2_a
Section VB96_2_b
Section VB96_2_c
Section VB96_2_d
Section VB96_2_e
Section VB96_2_f
Subsection VB96_2_f_f1
Subsection VB96_2_f_f2
Subsection VB96_2_f_f3
Subsection VB96_2_f_f4
Subsection VB96_2_f_f5
Subsection VB96_2_f_f6
Section VB96_2_g
Subsection VB96_2_g_g1
Chapter VB96_3
Document VB97
Chapter VB97_1
Section VB97_1_a
Subsection VB97_1_a_a1
Chapter VB97_2
Section VB97_2_a
Section VB97_2_b
Section VB97_2_c
Section VB97_2_d
Section VB97_2_e
Section VB97_2_f
Section VB97_2_g
Section VB97_2_h
Section VB97_2_i
Section VB97_2_j
Section VB97_2_k
Section VB97_2_l
Section VB97_2_m
Section VB97_2_n
Section VB97_2_o
Subsection VB97_2_o_o1
Subsection VB97_2_o_o2
Subsection VB97_2_o_o3
Section VB97_2_p
Section VB97_2_q
Subsection VB97_2_q_q1
Subsection VB97_2_q_q2
Section VB97_2_r
Subsection VB97_2_r_r1
Section VB97_2_s


Creating Neo4j trees:  97%|█████████▋| 97/100 [00:08<00:00, 11.16doc/s]

Section VB97_2_t
Section VB97_2_u
Subsection VB97_2_u_u1
Subsection VB97_2_u_u2
Section VB97_2_v
Document VB98
Chapter VB98_1
Section VB98_1_a
Chapter VB98_2
Section VB98_2_a
Section VB98_2_b
Section VB98_2_c
Section VB98_2_d
Subsection VB98_2_d_d1
Subsection VB98_2_d_d2
Subsection VB98_2_d_d3
Subsection VB98_2_d_d4
Subsection VB98_2_d_d5
Subsection VB98_2_d_d6
Chapter VB98_3
Section VB98_3_a
Subsection VB98_3_a_a1
Subsection VB98_3_a_a2
Subsection VB98_3_a_a3
Section VB98_3_b
Subsection VB98_3_b_b1
Subsection VB98_3_b_b2
Subsection VB98_3_b_b3
Subsection VB98_3_b_b4
Subsection VB98_3_b_b5
Section VB98_3_c
Subsection VB98_3_c_c1
Section VB98_3_d
Subsection VB98_3_d_d1
Subsection VB98_3_d_d2
Section VB98_3_e
Subsection VB98_3_e_e1
Subsection VB98_3_e_e2
Document VB99
Chapter VB99_1
Section VB99_1_a


Creating Neo4j trees: 100%|██████████| 100/100 [00:08<00:00, 11.66doc/s]

Chapter VB99_2
Section VB99_2_a
Section VB99_2_b
Subsection VB99_2_b_b1
Subsection VB99_2_b_b2
Section VB99_2_c
Subsection VB99_2_c_c1
Subsection VB99_2_c_c2
Chapter VB99_3
Section VB99_3_a
Section VB99_3_b
Document VB100
Chapter VB100_1
Section VB100_1_a
Section VB100_1_b
Chapter VB100_2
Section VB100_2_a
Section VB100_2_b
Subsection VB100_2_b_b1
Subsection VB100_2_b_b2
Subsection VB100_2_b_b3
Chapter VB100_3
Complete create all tree!
